# تدريب موديل تسميع مُتقِن 🎤

**قبل ما تبدأ:** من Runtime → Change runtime type → اختار **GPU (T4)**.

الدورة كلها هنا: الـWorker بيدّي Colab داتا التسميع، وColab بيرجّعله الموديل
الجديد — تطبيق اليوزر ملوش أي دعوة بالقصة دي (هو بس بيرفع الداتا وبينزّل الموديل الجاهز).

شغّل الخلايا بالترتيب — هيطلب منك مفتاح الـWorker مرة واحدة (نفس SYNC_TOKEN).

In [ ]:
# ١) تثبيت NeMo وأدوات التصدير (~١٠ دقايق)
!pip install -q "nemo_toolkit[asr]" onnx onnxruntime huggingface_hub requests

In [ ]:
# ٢) المفتاح + جلب السكريبتات من الريبو
import os
from getpass import getpass
os.environ['WORKER'] = 'https://mutqin-collector.mutqin.workers.dev'
os.environ['MUTQIN_KEY'] = getpass('مفتاح الـWorker ‏(MUTQIN_KEY): ').strip()
BASE = 'https://raw.githubusercontent.com/abdoadel123/mutqin-resources/main/train'
for f in ['prep.py', 'finetune.py', 'export_int8.py']:
    !wget -q {BASE}/{f} -O /content/{f}
print('جاهز')

In [ ]:
# ٣) تجهيز الداتا: سحب كل جلسات المساهمة من الـWorker → wav + manifest
!cd /content && python prep.py

In [ ]:
# ٤) التدريب (الافتراضي ٢٠ دورة — عدّل EPOCHS لو حبيت)
%env DATA=/content/train-out
!cd /content && python finetune.py

In [ ]:
# ٥) تصدير ONNX + ضغط INT8 + نشر تلقائي عبر الـWorker
# (المقاطع أولًا والمانيفست آخر حاجة — التطبيقات تكتشف التحديث وتستأذن المستخدم)
!cd /content && python export_int8.py